In [ ]:
# ============================================================
# EXPERIMENT 5A: DECISION TREE CLASSIFICATION FROM SCRATCH
# ============================================================
import numpy as np
from collections import Counter

class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature, self.threshold, self.left, self.right, self.value = feature, threshold, left, right, value

class DecisionTreeClassifierScratch:
    def __init__(self, max_depth=3, min_samples_split=2):
        self.max_depth, self.min_samples_split, self.root = max_depth, min_samples_split, None

    def entropy(self, y):
        if len(y) == 0: return 0
        probs = np.bincount(y)
        probs = probs[probs > 0] / len(y)
        return -np.sum(probs * np.log2(probs))

    def information_gain(self, y, y_left, y_right):
        n, n_l, n_r = len(y), len(y_left), len(y_right)
        if n_l == 0 or n_r == 0: return 0
        return self.entropy(y) - ((n_l / n) * self.entropy(y_left) + (n_r / n) * self.entropy(y_right))

    def best_split(self, X, y):
        best_gain, best_feat, best_thresh = -1, None, None
        for feat in range(X.shape[1]):
            for thresh in np.unique(X[:, feat]):
                y_l, y_r = y[X[:, feat] <= thresh], y[X[:, feat] > thresh]
                if len(y_l) == 0 or len(y_r) == 0: continue
                gain = self.information_gain(y, y_l, y_r)
                if gain > best_gain:
                    best_gain, best_feat, best_thresh = gain, feat, thresh
        return best_feat, best_thresh, best_gain

    def build_tree(self, X, y, depth=0):
        if depth >= self.max_depth or len(np.unique(y)) == 1 or len(y) < self.min_samples_split:
            return Node(value=Counter(y).most_common(1)[0][0])
        
        feat, thresh, gain = self.best_split(X, y)
        if feat is None or gain <= 0:
            return Node(value=Counter(y).most_common(1)[0][0])
        
        left_mask = X[:, feat] <= thresh
        return Node(feat, thresh, 
                    self.build_tree(X[left_mask], y[left_mask], depth + 1),
                    self.build_tree(X[~left_mask], y[~left_mask], depth + 1))

    def fit(self, X, y):
        self.root = self.build_tree(np.array(X, dtype=float), np.array(y, dtype=int))

    def predict_one(self, x, node):
        if node.value is not None: return node.value
        return self.predict_one(x, node.left if x[node.feature] <= node.threshold else node.right)

    def predict(self, X):
        return np.array([self.predict_one(x, self.root) for x in np.array(X, dtype=float)])

# Dataset and Train-Test Split
X = np.array([[1, 55], [2, 60], [3, 65], [4, 70], [5, 75], [6, 80], [7, 85], [8, 90]])
y = np.array([0, 0, 0, 1, 1, 1, 1, 1])
X_train, y_train, X_test, y_test = X[:6], y[:6], X[6:], y[6:]

print(f"Training Data:\n{X_train}\n\nTraining Labels:\n{y_train}\n\nTesting Data:\n{X_test}\n\nTesting Labels:\n{y_test}")

# Model Initialization and Initial Entropy
tree = DecisionTreeClassifierScratch(max_depth=3)
print(f"\nInitial Entropy:\n{round(tree.entropy(y_train), 4)}")

# Best Split Calculation
feat, thresh, gain = tree.best_split(X_train, y_train)
print(f"\nBest Split\n------------------\nFeature Index : {feat}\nThreshold : {thresh}\nInformation Gain : {round(gain, 4)}")

# Train, Predict, and Evaluate
tree.fit(X_train, y_train)
y_pred = tree.predict(X_test)

acc = np.mean(y_test == y_pred)
print(f"\nActual Labels :\n{y_test}\n\nPredicted Labels :\n{y_pred}\n\nAccuracy : {round(acc, 4)}\nAccuracy (%): {round(acc * 100, 2)}")

In [ ]:
# ============================================================
# EXPERIMENT 5B: DECISION TREE REGRESSION FROM SCRATCH
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

# Step 1 & 2: Create Synthetic Dataset and Train-Test Split
X = np.array([[1.0], [1.5], [2.0], [2.5], [3.0], [3.5], [4.0], [4.5], [5.0], [5.5],
              [6.0], [6.5], [7.0], [7.5], [8.0], [8.5], [9.0], [9.5], [10.0], [10.5]])
y = np.array([35, 38, 40, 45, 50, 52, 55, 60, 65, 68, 70, 72, 75, 78, 85, 86, 88, 90, 95, 96], dtype=float)
print(f"Dataset Shape: {X.shape}")

np.random.seed(42)
idx = np.random.permutation(len(X))
split = int(0.8 * len(X))
X_train, y_train = X[idx[:split]], y[idx[:split]]
X_test, y_test = X[idx[split:]], y[idx[split:]]
print(f"Training Samples: {len(X_train)}\nTesting Samples: {len(X_test)}")

# Step 3 & 4: Tree Node and Decision Tree Regressor Classes
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature, self.threshold, self.left, self.right, self.value = feature, threshold, left, right, value

class DecisionTreeRegressorScratch:
    def __init__(self, max_depth=3, min_samples_split=2):
        self.max_depth, self.min_samples_split, self.root = max_depth, min_samples_split, None

    def mse(self, y):
        return 0 if len(y) == 0 else np.mean((y - np.mean(y)) ** 2)

    def weighted_mse(self, y_l, y_r):
        n = len(y_l) + len(y_r)
        return 0 if n == 0 else (len(y_l)/n)*self.mse(y_l) + (len(y_r)/n)*self.mse(y_r)

    def best_split(self, X, y):
        best_mse, best_feat, best_thr = np.inf, None, None
        for f in range(X.shape[1]):
            for thr in np.unique(X[:, f]):
                y_l, y_r = y[X[:, f] <= thr], y[X[:, f] > thr]
                if len(y_l) == 0 or len(y_r) == 0: continue
                w_mse = self.weighted_mse(y_l, y_r)
                if w_mse < best_mse:
                    best_mse, best_feat, best_thr = w_mse, f, thr
        return best_feat, best_thr, best_mse

    def build_tree(self, X, y, depth=0):
        if len(y) == 0: return Node(value=0.0)
        if depth >= self.max_depth or len(y) < self.min_samples_split or np.all(y == y[0]):
            return Node(value=np.mean(y))
        feat, thr, _ = self.best_split(X, y)
        if feat is None: return Node(value=np.mean(y))
        left = self.build_tree(X[X[:, feat] <= thr], y[X[:, feat] <= thr], depth + 1)
        right = self.build_tree(X[X[:, feat] > thr], y[X[:, feat] > thr], depth + 1)
        return Node(feature=feat, threshold=thr, left=left, right=right)

    def fit(self, X, y): self.root = self.build_tree(X, y)

    def predict_one(self, x, node):
        if node.value is not None: return node.value
        return self.predict_one(x, node.left if x[node.feature] <= node.threshold else node.right)

    def predict(self, X): return np.array([self.predict_one(x, self.root) for x in X])

# Step 5, 6, 7: Train Model, Display Tree, and Make Predictions
model = DecisionTreeRegressorScratch(max_depth=3, min_samples_split=2)
model.fit(X_train, y_train)
print("\nDecision Tree Regression Model Trained Successfully.")

def print_tree(node, indent=""):
    if node.value is not None:
        print(f"{indent}Leaf -> Prediction: {round(node.value, 4)}")
        return
    print(f"{indent}X[{node.feature}] <= {node.threshold}\n{indent}Left:")
    print_tree(node.left, indent + "  ")
    print(f"{indent}Right:")
    print_tree(node.right, indent + "  ")

print("\n===================================\nTREE STRUCTURE\n===================================")
print_tree(model.root)

y_pred = model.predict(X_test)
print("\n===================================\nACTUAL VS PREDICTED\n===================================")
print("Hours\tActual\tPredicted")
for i in range(len(y_test)):
    print(f"{X_test[i, 0]:.1f}\t{y_test[i]:.2f}\t{y_pred[i]:.2f}")

# Step 8, 9, 10: Performance Measures, New Prediction, and Plot
errors = y_test - y_pred
mae, mse = np.mean(np.abs(errors)), np.mean(errors ** 2)
rmse, r2 = np.sqrt(mse), 1 - (np.sum(errors**2) / np.sum((y_test - np.mean(y_test))**2))

print("\n===================================\nDECISION TREE REGRESSION PERFORMANCE\n===================================")
print(f"MAE : {round(mae, 4)}\nMSE : {round(mse, 4)}\nRMSE: {round(rmse, 4)}\nR2  : {round(r2, 4)}")

new_hours = 6.2
new_pred = model.predict(np.array([[new_hours]]))[0]
print(f"\n===================================\nNEW PREDICTION\n===================================")
print(f"Study Hours: {new_hours}\nPredicted Marks: {round(new_pred, 2)}")

X_grid = np.linspace(X.min(), X.max(), 300).reshape(-1, 1)
plt.figure(figsize=(8, 5))
plt.scatter(X, y, label="Actual Data")
plt.plot(X_grid, model.predict(X_grid), label="Decision Tree Prediction")
plt.scatter(new_hours, new_pred, marker="x", s=120, label="New Prediction")
plt.xlabel("Study Hours"), plt.ylabel("Marks")
plt.title("Decision Tree Regression From Scratch")
plt.legend(), plt.grid(True), plt.tight_layout(), plt.show()